# Univariate Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/02-exploratory-data-analysis/02_univariate_analysis.ipynb)

## Learning Objectives
- Understand single-variable analysis techniques
- Master visualization methods for different data types
- Learn to interpret distributions and summary statistics
- Identify patterns, anomalies, and data quality issues

---

## 1. What is Univariate Analysis?

**Univariate Analysis** = Analyzing **ONE variable at a time**

**Goals:**
- 📊 Understand the distribution (shape, spread, center)
- 🔍 Detect outliers and anomalies
- 📈 Identify patterns (skewness, modality)
- 🎯 Assess data quality

**Two main types:**
1. **Numerical variables**: Continuous (age, price) or discrete (count, rating)
2. **Categorical variables**: Nominal (color, city) or ordinal (rating, education level)

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

## 2. Dataset: Customer Churn Analysis

Let's analyze customer data from a telecom company.

In [ ]:
# Create synthetic customer dataset
np.random.seed(42)
n = 1000

data = {
    'customer_id': range(1, n+1),
    'age': np.random.normal(40, 15, n).clip(18, 80).astype(int),
    'tenure_months': np.random.exponential(24, n).clip(1, 72).astype(int),
    'monthly_charges': np.random.normal(65, 20, n).clip(20, 150).round(2),
    'total_spend': np.random.gamma(50, 100, n).clip(100, 8000).round(2),
    'num_products': np.random.choice([1, 2, 3, 4], n, p=[0.3, 0.4, 0.2, 0.1]),
    'contract_type': np.random.choice(['Month-to-Month', 'One Year', 'Two Year'], n, p=[0.5, 0.3, 0.2]),
    'payment_method': np.random.choice(['Credit Card', 'Bank Transfer', 'Electronic Check', 'Cash'], n, p=[0.3, 0.25, 0.35, 0.1]),
    'internet_service': np.random.choice(['DSL', 'Fiber', 'No'], n, p=[0.4, 0.45, 0.15]),
    'customer_service_calls': np.random.poisson(2, n),
    'churn': np.random.choice(['Yes', 'No'], n, p=[0.27, 0.73])
}

df = pd.DataFrame(data)

# Add some missing values
missing_idx = np.random.choice(n, 30, replace=False)
df.loc[missing_idx[:15], 'monthly_charges'] = np.nan
df.loc[missing_idx[15:], 'tenure_months'] = np.nan

print("✅ Customer dataset created!")
print(f"Shape: {df.shape}")
df.head()

## 3. Numerical Variables Analysis

### 3.1 Central Tendency Measures

In [ ]:
# Example: Age analysis
age = df['age']

print("👤 Age Statistics:")
print(f"Mean (average): {age.mean():.1f} years")
print(f"Median (middle): {age.median():.1f} years")
print(f"Mode (most common): {age.mode()[0]} years")

print(f"\n📏 Spread:")
print(f"Standard Deviation: {age.std():.1f} years")
print(f"Range: {age.min()} - {age.max()} years")
print(f"IQR (Q3-Q1): {age.quantile(0.75) - age.quantile(0.25):.1f} years")

print(f"\n📊 Quartiles:")
print(f"25th percentile (Q1): {age.quantile(0.25):.1f}")
print(f"50th percentile (Q2/Median): {age.quantile(0.50):.1f}")
print(f"75th percentile (Q3): {age.quantile(0.75):.1f}")

### 3.2 Distribution Shape

In [ ]:
# Skewness and Kurtosis
print("📐 Distribution Shape:")
print(f"Skewness: {age.skew():.2f}")
print(f"Kurtosis: {age.kurtosis():.2f}")

# Interpretation
skew = age.skew()
if abs(skew) < 0.5:
    print("\n✅ Distribution is approximately symmetric")
elif skew > 0.5:
    print("\n➡️ Distribution is right-skewed (tail on the right)")
else:
    print("\n⬅️ Distribution is left-skewed (tail on the left)")

kurt = age.kurtosis()
if abs(kurt) < 0.5:
    print("✅ Distribution has normal tails (mesokurtic)")
elif kurt > 0.5:
    print("⬆️ Distribution has heavy tails (leptokurtic - more outliers)")
else:
    print("⬇️ Distribution has light tails (platykurtic - fewer outliers)")

### 3.3 Visualization Techniques

In [ ]:
# Histogram
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogram
axes[0, 0].hist(age, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].axvline(age.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {age.mean():.1f}')
axes[0, 0].axvline(age.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {age.median():.1f}')
axes[0, 0].set_title('Histogram of Age', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Box Plot
axes[0, 1].boxplot(age, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
axes[0, 1].set_title('Box Plot of Age', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Age')
axes[0, 1].grid(alpha=0.3)

# 3. KDE (Kernel Density Estimate)
age.plot(kind='density', ax=axes[1, 0], color='purple', linewidth=2)
axes[1, 0].fill_between(age.plot(kind='density').get_lines()[0].get_data()[0],
                         age.plot(kind='density').get_lines()[0].get_data()[1],
                         alpha=0.3, color='purple')
axes[1, 0].set_title('Density Plot of Age', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Age')
axes[1, 0].set_ylabel('Density')
axes[1, 0].grid(alpha=0.3)

# 4. Violin Plot
parts = axes[1, 1].violinplot([age], vert=True, showmeans=True, showmedians=True)
axes[1, 1].set_title('Violin Plot of Age', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Age')
axes[1, 1].set_xticks([1])
axes[1, 1].set_xticklabels(['Age'])
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.4 Outlier Detection

In [ ]:
# Method 1: IQR Method
Q1 = age.quantile(0.25)
Q3 = age.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = age[(age < lower_bound) | (age > upper_bound)]

print("🔍 Outlier Detection (IQR Method):")
print(f"Lower Bound: {lower_bound:.1f}")
print(f"Upper Bound: {upper_bound:.1f}")
print(f"Outliers Found: {len(outliers_iqr)} ({len(outliers_iqr) / len(age) * 100:.1f}%)")

if len(outliers_iqr) > 0:
    print(f"\nOutlier Values: {sorted(outliers_iqr.values)[:10]}...")

In [ ]:
# Method 2: Z-Score Method
z_scores = np.abs(stats.zscore(age))
outliers_z = age[z_scores > 3]

print("\n🔍 Outlier Detection (Z-Score Method):")
print(f"Outliers Found (|Z| > 3): {len(outliers_z)} ({len(outliers_z) / len(age) * 100:.1f}%)")

if len(outliers_z) > 0:
    print(f"Outlier Values: {sorted(outliers_z.values)}")

### 3.5 Normality Test

In [ ]:
# Shapiro-Wilk Test
statistic, p_value = stats.shapiro(age.sample(min(5000, len(age))))  # Sample for large datasets

print("📊 Normality Test (Shapiro-Wilk):")
print(f"Test Statistic: {statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value > 0.05:
    print("\n✅ Data appears to be normally distributed (p > 0.05)")
else:
    print("\n❌ Data does NOT appear to be normally distributed (p <= 0.05)")

# Q-Q Plot
plt.figure(figsize=(8, 6))
stats.probplot(age, dist="norm", plot=plt)
plt.title('Q-Q Plot: Age vs Normal Distribution', fontsize=12, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 If points follow the red line closely → Normal distribution")
print("   If points deviate → Non-normal distribution")

### 3.6 Analyze All Numerical Columns

In [ ]:
# Get numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols.remove('customer_id')  # Remove ID column

# Summary table
summary_stats = []

for col in numerical_cols:
    data_col = df[col].dropna()
    
    summary_stats.append({
        'Variable': col,
        'Count': len(data_col),
        'Missing': df[col].isnull().sum(),
        'Mean': data_col.mean(),
        'Median': data_col.median(),
        'Std': data_col.std(),
        'Min': data_col.min(),
        'Max': data_col.max(),
        'Skewness': data_col.skew(),
        'Kurtosis': data_col.kurtosis()
    })

summary_df = pd.DataFrame(summary_stats)
print("📊 Numerical Variables Summary:")
summary_df.round(2)

In [ ]:
# Visualize all numerical distributions
n_cols = len(numerical_cols)
n_rows = (n_cols + 2) // 3

fig, axes = plt.subplots(n_rows, 3, figsize=(15, n_rows * 4))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    data_col = df[col].dropna()
    
    axes[idx].hist(data_col, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[idx].axvline(data_col.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
    axes[idx].axvline(data_col.median(), color='green', linestyle='--', linewidth=2, label='Median')
    axes[idx].set_title(f'{col}\n(Skew: {data_col.skew():.2f})', fontsize=10, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend(fontsize=8)
    axes[idx].grid(alpha=0.3)

# Hide unused subplots
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 4. Categorical Variables Analysis

### 4.1 Frequency Distribution

In [ ]:
# Example: Contract Type
contract_counts = df['contract_type'].value_counts()
contract_pct = df['contract_type'].value_counts(normalize=True) * 100

print("📋 Contract Type Distribution:")
print("\nCounts:")
print(contract_counts)
print("\nPercentages:")
print(contract_pct.round(2))

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar Chart
contract_counts.plot(kind='bar', ax=axes[0], color='teal', edgecolor='black', alpha=0.7)
axes[0].set_title('Contract Type - Bar Chart', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Contract Type')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)
axes[0].grid(alpha=0.3, axis='y')

# Pie Chart
axes[1].pie(contract_counts, labels=contract_counts.index, autopct='%1.1f%%',
            startangle=90, colors=['#ff9999','#66b3ff','#99ff99'])
axes[1].set_title('Contract Type - Pie Chart', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 4.2 Analyze All Categorical Variables

In [ ]:
# Get categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    print("\n" + "="*60)
    print(f"📊 {col.upper()} ANALYSIS")
    print("="*60)
    
    # Value counts
    counts = df[col].value_counts()
    pct = df[col].value_counts(normalize=True) * 100
    
    summary = pd.DataFrame({
        'Count': counts,
        'Percentage': pct
    })
    
    print(summary.round(2))
    
    # Check for imbalance
    max_pct = pct.max()
    if max_pct > 70:
        print(f"\n⚠️ Highly imbalanced! Dominant class: {pct.idxmax()} ({max_pct:.1f}%)")
    elif max_pct > 50:
        print(f"\n⚡ Moderately imbalanced. Dominant class: {pct.idxmax()} ({max_pct:.1f}%)")
    else:
        print(f"\n✅ Fairly balanced distribution")
    
    # Unique values
    n_unique = df[col].nunique()
    print(f"\n🔢 Unique Values: {n_unique}")
    
    # Missing values
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        print(f"❓ Missing Values: {n_missing} ({n_missing / len(df) * 100:.1f}%)")

In [ ]:
# Visualize all categorical variables
n_cat = len(categorical_cols)
n_rows_cat = (n_cat + 1) // 2

fig, axes = plt.subplots(n_rows_cat, 2, figsize=(14, n_rows_cat * 4))
axes = axes.flatten()

for idx, col in enumerate(categorical_cols):
    counts = df[col].value_counts()
    
    counts.plot(kind='bar', ax=axes[idx], color='coral', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col} Distribution', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45, ha='right')
    axes[idx].grid(alpha=0.3, axis='y')
    
    # Add value labels on bars
    for container in axes[idx].containers:
        axes[idx].bar_label(container, fontsize=9)

# Hide unused subplot if odd number
if n_cat % 2 != 0:
    axes[-1].axis('off')

plt.tight_layout()
plt.show()

## 5. Missing Value Analysis

In [ ]:
# Missing value summary
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Pct': (df.isnull().sum().values / len(df) * 100).round(2)
}).sort_values('Missing_Count', ascending=False)

missing_data = missing_data[missing_data['Missing_Count'] > 0]

if len(missing_data) > 0:
    print("❓ Columns with Missing Values:")
    print(missing_data.to_string(index=False))
    
    # Visualize
    plt.figure(figsize=(10, 5))
    plt.bar(missing_data['Column'], missing_data['Missing_Pct'], color='salmon', edgecolor='black', alpha=0.7)
    plt.title('Missing Values by Column', fontsize=12, fontweight='bold')
    plt.xlabel('Column')
    plt.ylabel('Missing Percentage (%)')
    plt.xticks(rotation=45, ha='right')
    plt.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print("✅ No missing values found!")

## 6. Univariate Analysis Decision Tree

```
What type of variable?
    |
    ├─ NUMERICAL
    │   ├─ Summary stats (mean, median, std, quartiles)
    │   ├─ Distribution plots (histogram, box plot, KDE)
    │   ├─ Check skewness & kurtosis
    │   ├─ Detect outliers (IQR or Z-score)
    │   └─ Test normality (Shapiro-Wilk, Q-Q plot)
    │
    └─ CATEGORICAL
        ├─ Value counts & percentages
        ├─ Bar chart or pie chart
        ├─ Check for imbalance
        └─ Count unique values
```

## 7. Your Turn! 💪

**Exercise**: Pick 2 variables from the dataset (1 numerical, 1 categorical) and:
1. Calculate summary statistics
2. Create at least 2 visualizations
3. Identify any patterns or anomalies
4. Write 2-3 insights about each variable

In [ ]:
# Your code here

---

## Key Takeaways 🎯

1. **Numerical variables**: Use histograms, box plots, summary stats, outlier detection
2. **Categorical variables**: Use bar charts, value counts, check for class imbalance
3. **Always visualize**: Numbers alone can hide important patterns
4. **Check distribution shape**: Skewness and normality affect modeling choices
5. **Outliers aren't always errors**: Investigate before removing

**Next**: Learn bivariate analysis to explore relationships between variables! 🔗